### ⚙️ Inicialización de la Sesión de Spark

Crearemos la sesión con un driver específico para la lectura de la base de datos. Para establecer la conexión, introduciremos:
* `URL`
* `Usuario` y `Contraseña`
* `Driver`

💡 **Nota de rendimiento:** Esto nos permite crear los DataFrames sin cargar los datos directamente en la RAM, difiriendo la ejecución hasta que se realice una acción.

In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

# 1. SETUP CON EL DRIVER DE POSTGRESQL
# Añadimos la configuración 'spark.jars.packages' apuntando a la versión estable del driver.
spark = SparkSession.builder \
    .appName("M5_JDBC_Connection") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .master("local[*]") \
    .getOrCreate()

print("Sesión iniciada. Descargando driver JDBC si es necesario...")

# 2. CONFIGURACIÓN DE LA CONEXIÓN
# Asumiendo que tu BD está en localhost y el puerto por defecto es 5432
db_url = "jdbc:postgresql://localhost:5432/m5_db"

# IMPORTANTE: Cambia 'tu_usuario' y 'tu_contraseña' por los que usas para entrar a pgAdmin
db_properties = {
    "user": "postgres",         # Usuario por defecto, cámbialo si es distinto
    "password": "9794", 
    "driver": "org.postgresql.Driver"
}

# 3. EXTRACCIÓN DE DATOS (Lectura distribuida)
print("Conectando a PostgreSQL y leyendo tablas...")

# Spark lee los metadatos y crea los DataFrames, pero no carga toda la data 
# a la memoria RAM hasta que haces una acción (como .show() o .count())
df_calendar = spark.read.jdbc(url=db_url, table="stg_calendar", properties=db_properties)
df_prices = spark.read.jdbc(url=db_url, table="stg_prices", properties=db_properties)
df_sales = spark.read.jdbc(url=db_url, table="stg_sales", properties=db_properties)

# 4. VERIFICACIÓN
print("¡Tablas cargadas exitosamente como DataFrames de PySpark!")
print("Muestra de stg_sales:")
df_sales.show(5)

d:\Usuarios\Escritorio\M5\m5-forecasting-project\venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Sesión iniciada. Descargando driver JDBC si es necesario...
Conectando a PostgreSQL y leyendo tablas...
¡Tablas cargadas exitosamente como DataFrames de PySpark!
Muestra de stg_sales:
+--------------------+-------------+---------+-------+--------+--------+-----+---------+
|                  id|      item_id|  dept_id| cat_id|store_id|state_id|    d|sales_qty|
+--------------------+-------------+---------+-------+--------+--------+-----+---------+
|HOBBIES_1_220_WI_...|HOBBIES_1_220|HOBBIES_1|HOBBIES|    WI_2|      WI|d_512|        2|
|HOBBIES_1_221_WI_...|HOBBIES_1_221|HOBBIES_1|HOBBIES|    WI_2|      WI|d_512|        0|
|HOBBIES_1_223_WI_...|HOBBIES_1_223|HOBBIES_1|HOBBIES|    WI_2|      WI|d_512|        0|
|HOBBIES_1_224_WI_...|HOBBIES_1_224|HOBBIES_1|HOBBIES|    WI_2|      WI|d_512|        0|
|HOBBIES_1_225_WI_...|HOBBIES_1_225|HOBBIES_1|HOBBIES|    WI_2|      WI|d_512|        0|
+--------------------+-------------+---------+-------+--------+--------+-----+---------+
only showing to

### 🔄 Asignación de Tipos de Datos (Casting)

Ahora ajustaremos el esquema de nuestro DataFrame asignando los tipos correspondientes a cada columna:
* **Ventas** ➔ `Integer`
* **Fecha** ➔ `DateType`
* **Precios** ➔ `Float`

💡 **Verificación:** Al finalizar estas conversiones, comprobaremos el esquema para asegurarnos de que Spark ha recibido y aplicado esta información correctamente.

In [2]:
from pyspark.sql.types import IntegerType, DateType, StringType, FloatType

print("Aplicando esquemas estrictos para optimizar memoria...")

# 1. Tipado de stg_sales (La tabla más pesada)
# Casteamos las ventas a Integer para reducir la RAM a la mitad (de 64 a 32 bits).
df_sales_typed = df_sales \
    .withColumn("sales_qty", df_sales["sales_qty"].cast(IntegerType())) \
    .withColumn("d", df_sales["d"].cast(StringType()))

# 2. Tipado de stg_calendar
# Fundamental para Time Series: Asegurar que 'date' es DateType real.
df_calendar_typed = df_calendar \
    .withColumn("date", df_calendar["date"].cast(DateType())) \
    .withColumn("wm_yr_wk", df_calendar["wm_yr_wk"].cast(IntegerType()))

# 3. Tipado de stg_prices
# Los precios los manejamos como Float (32 bits) en lugar de Double (64 bits)
df_prices_typed = df_prices \
    .withColumn("sell_price", df_prices["sell_price"].cast(FloatType())) \
    .withColumn("wm_yr_wk", df_prices["wm_yr_wk"].cast(IntegerType()))

print("¡Esquemas optimizados exitosamente!")

# Verificamos que Spark ha entendido los nuevos tipos
df_sales_typed.printSchema()

Aplicando esquemas estrictos para optimizar memoria...
¡Esquemas optimizados exitosamente!
root
 |-- id: string (nullable = true)
 |-- item_id: string (nullable = true)
 |-- dept_id: string (nullable = true)
 |-- cat_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- state_id: string (nullable = true)
 |-- d: string (nullable = true)
 |-- sales_qty: integer (nullable = true)



### 🧹 Limpieza de Datos: Nulos y Atípicos

Pasamos a la fase de calidad de datos, donde nos centraremos en analizar y corregir nuestras variables numéricas clave. Nos enfocaremos en dos tareas principales:
* **Valores Nulos:** Identificación y gestión de datos faltantes en el dataset.
* **Valores Atípicos (Outliers):** Detección y tratamiento de registros con valores extremos o anómalos.
* **Columnas objetivo:** `Precios` y `Ventas`.

💡 **Nota analítica:** Tratar correctamente estos valores es un paso crítico para no alterar las distribuciones reales de las ventas y evitar sesgos en nuestro análisis posterior.

Finalmente contabilizamos las "filas fantasma" de cada una de las tablas.

In [3]:
import pyspark.sql.functions as F

print("Iniciando limpieza de nulos y valores atípicos...")

# 1. Tratamiento de Ventas (stg_sales_typed)
# Llenamos los nulos con 0 y eliminamos devoluciones/errores (ventas negativas)
df_sales_clean = df_sales_typed \
    .fillna({"sales_qty": 0}) \
    .filter(F.col("sales_qty") >= 0)

# 2. Tratamiento de Precios (stg_prices_typed)
# Eliminamos filas sin precio (producto inactivo) y filtramos precios negativos o a 0
df_prices_clean = df_prices_typed \
    .dropna(subset=["sell_price"]) \
    .filter(F.col("sell_price") > 0)

# 3. Tratamiento de Calendario (stg_calendar_typed)
# El calendario suele venir limpio, pero eliminamos cualquier fecha nula por seguridad
df_calendar_clean = df_calendar_typed \
    .dropna(subset=["date"])

print("¡Limpieza completada!")


print("Calculando volumetría antes y después de la limpieza (esto puede tardar unos segundos)...")

# 1. Conteo de la tabla original (tipada)
sales_before = df_sales_typed.count()
prices_before = df_prices_typed.count()
calendar_before = df_calendar_typed.count()

# 2. Conteo de la tabla limpia
sales_after = df_sales_clean.count()
prices_after = df_prices_clean.count()
calendar_after = df_calendar_clean.count()

# 3. Cálculo de la diferencia (Filas fantasma o atípicas eliminadas)
sales_removed = sales_before - sales_after
prices_removed = prices_before - prices_after
calendar_removed = calendar_before - calendar_after

# 4. Reporte de auditoría
print("-" * 40)
print("AUDITORÍA DE LIMPIEZA DE DATOS M5")
print("-" * 40)
print(f"Ventas (stg_sales):")
print(f"  - Filas originales: {sales_before:,}")
print(f"  - Filas limpias:    {sales_after:,}")
print(f"  - Filas eliminadas (negativas): {sales_removed:,} ({((sales_removed/sales_before)*100):.2f}%)\n")

print(f"Precios (stg_prices):")
print(f"  - Filas originales: {prices_before:,}")
print(f"  - Filas limpias:    {prices_after:,}")
print(f"  - Filas eliminadas (sin precio/0): {prices_removed:,} ({((prices_removed/prices_before)*100):.2f}%)\n")

print(f"Calendario (stg_calendar):")
print(f"  - Filas originales: {calendar_before:,}")
print(f"  - Filas limpias:    {calendar_after:,}")
print(f"  - Filas eliminadas (nulas): {calendar_removed:,} ({((calendar_removed/calendar_before)*100):.2f}%)")
print("-" * 40)

Iniciando limpieza de nulos y valores atípicos...
¡Limpieza completada!
Calculando volumetría antes y después de la limpieza (esto puede tardar unos segundos)...
----------------------------------------
AUDITORÍA DE LIMPIEZA DE DATOS M5
----------------------------------------
Ventas (stg_sales):
  - Filas originales: 58,327,370
  - Filas limpias:    58,327,370
  - Filas eliminadas (negativas): 0 (0.00%)

Precios (stg_prices):
  - Filas originales: 6,841,121
  - Filas limpias:    6,841,121
  - Filas eliminadas (sin precio/0): 0 (0.00%)

Calendario (stg_calendar):
  - Filas originales: 1,969
  - Filas limpias:    1,969
  - Filas eliminadas (nulas): 0 (0.00%)
----------------------------------------


### 🔗 Unión de Tablas (Joins)

Vamos a consolidar la información uniendo nuestros DataFrames en dos pasos lógicos y secuenciales:
* **Paso 1:** Cruzamos `Ventas` con `Calendario` ➔ *Objetivo: Identificar en qué semana exacta ocurrió cada transacción.*
* **Paso 2:** Cruzamos el resultado anterior con `Precios` ➔ *Objetivo: Asignar el precio histórico correspondiente a esa misma semana.*

💡 **Estrategia de cruce:** Realizar estas uniones de forma escalonada nos ayuda a mantener un control claro sobre la granularidad de los datos (nivel semanal) y previene la duplicación accidental de registros.

In [4]:
import pyspark.sql.functions as F

print("Iniciando la construcción de la Tabla Maestra (Joins)...")

# 1. Primer Join: Ventas + Calendario
# Clave en común: La columna 'd' (ej. 'd_1', 'd_2')
# Usamos F.broadcast() porque el calendario es muy pequeño.
# Usamos un 'left' join para mantener todas nuestras ventas intactas.
df_sales_cal = df_sales_clean.join(
    F.broadcast(df_calendar_clean),
    on="d",
    how="left"
)

# 2. Segundo Join: Resultado Anterior + Precios
# Claves en común: Tienda, Producto y Semana (store_id, item_id, wm_yr_wk)
# Aquí no usamos broadcast porque la tabla de precios es mediana (casi 7M de filas).
df_master = df_sales_cal.join(
    df_prices_clean,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left"
)

print("¡Tabla Maestra generada en memoria!")

# Mostramos el esquema resultante y unas filas de muestra para validar
df_master.printSchema()
df_master.show(5)

Iniciando la construcción de la Tabla Maestra (Joins)...
¡Tabla Maestra generada en memoria!
root
 |-- store_id: string (nullable = true)
 |-- item_id: string (nullable = true)
 |-- wm_yr_wk: integer (nullable = true)
 |-- d: string (nullable = true)
 |-- id: string (nullable = true)
 |-- dept_id: string (nullable = true)
 |-- cat_id: string (nullable = true)
 |-- state_id: string (nullable = true)
 |-- sales_qty: integer (nullable = false)
 |-- date: date (nullable = true)
 |-- weekday: string (nullable = true)
 |-- wday: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- event_name_1: string (nullable = true)
 |-- event_type_1: string (nullable = true)
 |-- event_name_2: string (nullable = true)
 |-- event_type_2: string (nullable = true)
 |-- snap_ca: integer (nullable = true)
 |-- snap_tx: integer (nullable = true)
 |-- snap_wi: integer (nullable = true)
 |-- sell_price: float (nullable = true)

+--------+-------------+--------

### ⚠️ Control de Volumetría: Prevención de *Fan-out*

El mayor riesgo al realizar cruces de tablas es sufrir una **Explosión Cartesiana** (o *Fan-out*). Para entender la gravedad de este problema en nuestro caso:
* **El origen:** Si por accidente la tabla de `Precios` tuviera dos precios distintos para el mismo producto en la misma semana, el `left join` duplicaría la fila de ventas para emparejarla con ambos.
* **El impacto:** Pasaríamos de tener 58 millones de filas a cientos de millones de forma silenciosa, arruinando por completo la precisión del modelo predictivo.

💡 **Validación rápida:** A continuación, ejecutaremos un conteo de filas sobre nuestro `df_master`. Si el cruce se ha realizado correctamente, la volumetría debe mantenerse idéntica a la de nuestra tabla base de ventas.

In [5]:
import pyspark.sql.functions as F

print("Validando volumetría con técnica de aislamiento (Tienda CA_1)...")

# 1. Aislamos una única tienda para hacer la prueba en memoria local
tienda_test = "CA_1"

# Filtramos las tablas base
df_sales_test = df_sales_clean.filter(F.col("store_id") == tienda_test)
df_prices_test = df_prices_clean.filter(F.col("store_id") == tienda_test)

# 2. Contamos las filas base de esa tienda
sales_test_count = df_sales_test.count()

# 3. Aplicamos la misma receta de Joins solo a esta muestra
df_test_cal = df_sales_test.join(
    F.broadcast(df_calendar_clean),
    on="d",
    how="left"
)

df_master_test = df_test_cal.join(
    df_prices_test,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left"
)

# 4. Contamos el resultado cruzado
master_test_count = df_master_test.count()

# 5. Auditoría
print("-" * 40)
print(f"AUDITORÍA DE JOINS (MUESTRA: {tienda_test})")
print("-" * 40)
print(f"Filas limpias originales:  {sales_test_count:,}")
print(f"Filas tras el cruce:       {master_test_count:,}")
print("-" * 40)

if master_test_count == sales_test_count:
    print("✅ ¡Éxito absoluto! La lógica del JOIN es perfecta. No hay explosión cartesiana.")
else:
    print(f"❌ ALERTA: Diferencia detectada de {master_test_count - sales_test_count:,} filas.")

Validando volumetría con técnica de aislamiento (Tienda CA_1)...
----------------------------------------
AUDITORÍA DE JOINS (MUESTRA: CA_1)
----------------------------------------
Filas limpias originales:  5,832,737
Filas tras el cruce:       5,832,737
----------------------------------------
✅ ¡Éxito absoluto! La lógica del JOIN es perfecta. No hay explosión cartesiana.


In [6]:
import pyspark.sql.functions as F

print("Creando features temporales para el modelo...")

# Usamos withColumn para crear nuevas columnas basadas en la fecha original
df_master_features = df_master \
    .withColumn("is_weekend", F.when(F.dayofweek("date").isin([1, 7]), 1).otherwise(0)) \
    .withColumn("day_of_week", F.dayofweek("date")) \
    .withColumn("month", F.month("date")) \
    .withColumn("quarter", F.quarter("date")) \
    .withColumn("is_month_start", F.when(F.dayofmonth("date") == 1, 1).otherwise(0)) \
    .withColumn("is_month_end", F.when(F.dayofmonth("date") == F.last_day("date"), 1).otherwise(0))

print("¡Features temporales creadas correctamente!")

# Una muestra rápida para verificar que las columnas nuevas tienen sentido
df_master_features.select("date", "is_weekend", "month", "is_month_start").show(5)

Creando features temporales para el modelo...


{"ts": "2026-07-18 13:07:18.834", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[DATATYPE_MISMATCH.BINARY_OP_DIFF_TYPES] Cannot resolve \"(dayofmonth(date) = last_day(date))\" due to data type mismatch: the left and right operands of the binary operator have incompatible types (\"INT\" and \"DATE\"). SQLSTATE: 42K09", "context": {"file": "line 12 in cell [7]", "line": "", "fragment": "__eq__", "errorClass": "DATATYPE_MISMATCH.BINARY_OP_DIFF_TYPES"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o156.withColumn.\n: org.apache.spark.sql.AnalysisException: [DATATYPE_MISMATCH.BINARY_OP_DIFF_TYPES] Cannot resolve \"(dayofmonth(date) = last_day(date))\" due to data type mismatch: the left and right operands of the binary operator have incompatible types (\"INT\" and \"DATE\"). SQLSTATE: 42K09;\n'Project [store_id#22, item_id#19, wm_yr_wk#54, d#52, id#18, dept_id#20, cat_id#21, state_id#23, sales_qty#57, date#53, weekday#2, wday#3, month#

AnalysisException: [DATATYPE_MISMATCH.BINARY_OP_DIFF_TYPES] Cannot resolve "(dayofmonth(date) = last_day(date))" due to data type mismatch: the left and right operands of the binary operator have incompatible types ("INT" and "DATE"). SQLSTATE: 42K09;
'Project [store_id#22, item_id#19, wm_yr_wk#54, d#52, id#18, dept_id#20, cat_id#21, state_id#23, sales_qty#57, date#53, weekday#2, wday#3, month#241, year#5, event_name_1#7, event_type_1#8, event_name_2#9, event_type_2#10, snap_ca#11, snap_tx#12, snap_wi#13, sell_price#55, is_weekend#239, day_of_week#240, quarter#242, ... 2 more fields]
+- Project [store_id#22, item_id#19, wm_yr_wk#54, d#52, id#18, dept_id#20, cat_id#21, state_id#23, sales_qty#57, date#53, weekday#2, wday#3, month#241, year#5, event_name_1#7, event_type_1#8, event_name_2#9, event_type_2#10, snap_ca#11, snap_tx#12, snap_wi#13, sell_price#55, is_weekend#239, day_of_week#240, quarter#242, ... 1 more fields]
   +- Project [store_id#22, item_id#19, wm_yr_wk#54, d#52, id#18, de